# 02 - Estacionariedade e Fractional Differentiation

Implementação do método de Kaabar para encontrar o d ótimo via ADF test.

**IMPORTANTE**: FracDiff só em features de preço (não discretas).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import adfuller
import sys
sys.path.append('../src')
from fracdiff import fdiff

plt.style.use('dark_background')
DATA_DIR = Path('../data')

## 1. Carregar Dados

In [ ]:
df = pd.read_parquet(DATA_DIR / 'signals.parquet')
print(f'Total: {len(df)} sinais')

# Features de PREÇO (precisam de FracDiff)
price_features = ['entry_price', 'f_price_context']

# Features DISCRETAS (NÃO aplicar FracDiff)
discrete_features = ['f_imbalance', 'f_absorption', 'f_smart_money', 'score']

## 2. Teste ADF

In [ ]:
def adf_test(series, name=''):
    result = adfuller(series.dropna(), autolag='AIC')
    p_value = result[1]
    is_stationary = p_value < 0.05
    print(f'{name}: p={p_value:.4f} | {"✅" if is_stationary else "❌"}')
    return p_value, is_stationary

print('=== Features de Preço ===')
for col in price_features:
    if col in df.columns:
        adf_test(df[col], col)

print('\n=== Features Discretas (já estacionárias) ===')
for col in discrete_features:
    if col in df.columns:
        adf_test(df[col], col)

## 3. Find Optimal d

In [ ]:
def find_optimal_d(series, max_d=1.0, step=0.1, p_threshold=0.05):
    results = []
    for d in np.arange(0.0, max_d + step, step):
        try:
            if d == 0:
                diff_series = series
            else:
                diff_series = pd.Series(fdiff(series.values, d))
            diff_series = diff_series.replace([np.inf, -np.inf], np.nan).dropna()
            if len(diff_series) < 20:
                continue
            adf_result = adfuller(diff_series, autolag='AIC')
            results.append({'d': round(d, 2), 'p_value': adf_result[1], 'is_stat': adf_result[1] < p_threshold})
        except:
            continue
    results_df = pd.DataFrame(results)
    stationary = results_df[results_df['is_stat']]
    d_optimal = stationary['d'].min() if len(stationary) > 0 else max_d
    return d_optimal, results_df

optimal_d = {}
for col in price_features:
    if col not in df.columns:
        continue
    d, results = find_optimal_d(df[col].dropna())
    optimal_d[col] = d
    print(f'{col}: d ótimo = {d}')

In [ ]:
# Salvar
import json
with open(DATA_DIR / 'optimal_d.json', 'w') as f:
    json.dump(optimal_d, f, indent=2)
print(f'Salvo: {optimal_d}')